Run this then restart kernel to have the proper torch version

In [21]:
import sys
!{sys.executable} -m pip install torch==2.6.0 torchvision --index-url https://download.pytorch.org/whl/cu124 --force-reinstall --no-deps

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached torch-2.6.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (6.1 kB)
Using cached torch-2.6.0%2Bcu124-cp313-cp313-linux_x86_64.whl (768.4 MB)
Using cached torchvision-0.21.0%2Bcu124-cp313-cp313-linux_x86_64.whl (7.3 MB)
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.21.0+cu124
    Uninstalling torchvision-0.21.0+cu124:
      Successfully uninstalled torchvision-0.21.0+cu124
  Attempting uninstall: torch━━━━━━━━━━━━━━━━━━━ 0/2 [torchvision]
    Found existing installation: torch 2.6.0+cu1240/2 [torchvision]
    Uninstalling torch-2.6.0+cu124:0m╺━━━━━━━━━━━━━━━━━━━ 1/2 [torch]
      Successfully uninstalled torch-2.6.0+cu124━━━━━━━━━━━━━━ 1/2 [torch]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torch]32m1/2 [torch]


In [1]:
!sudo apt update && sudo apt install ffmpeg -y

!pip install whisper-openai
!pip install InaSpeechSegmenter
!pip install pyannote.audio
!pip install --upgrade transformers

Get:1 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:2 https://ppa.launchpadcontent.net/git-core/ppa/ubuntu noble InRelease [24.3 kB]
Get:3 https://apt.postgresql.org/pub/repos/apt noble-pgdg InRelease [180 kB]   
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]      
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]        
Get:7 https://ppa.launchpadcontent.net/git-core/ppa/ubuntu noble/main amd64 Packages [2,992 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]      
Get:9 https://apt.postgresql.org/pub/repos/apt noble-pgdg/main amd64 Packages [670 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble/universe amd64 Packages [19.3 MB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,333 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble/restricted amd64 P

In [1]:
import torch
torch.cuda.is_available()

True

### A ne run qu'une seule fois !

In [3]:
import whisper
import torch

torch.cuda.empty_cache()

whisper_model = whisper.load_model("medium", device="cuda") # remove device for CPU

100%|██████████████████████████████████████| 1.42G/1.42G [00:14<00:00, 107MiB/s]


In [ ]:
import os
import s3fs

#A RETIRER SYSTEMATIQUEMENT

#CREDENTIALS


#TOKEN Huggingface
token = ""


fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])


# Récupération des fichiers

s3_folder = "patin_art" #ski_alp

wav_files = fs.glob(f"s3://lab/PESSD/wav/{s3_folder}/*.wav")

for file in wav_files:
    fs.get(file, "data/"+(file.removeprefix(f"lab/PESSD/wav/{s3_folder}")))


# Audios à traiter
file_list = os.listdir("data")

In [ ]:
# Si sélection de fichiers
file_list = ["data/BHR_wav.wav"]

# Code final

In [ ]:
from pyannote.audio import Pipeline
from inaSpeechSegmenter import Segmenter
from inaSpeechSegmenter.export_funcs import seg2csv
import pandas as pd
import os
import whisper
import torch
import soundfile as sf 


# Pipeline pour diarization
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=token
)
# remove for CPU friendly
pipeline.to(torch.device("cuda"))

# Modèle InaSpeechSegmenter (genre)
seg_model = Segmenter()

# Liste des fichiers audio
audios = file_list  

#### BOUCLE PRINCIPALE
for audio_path in audios:

    # --- RESET DES LISTES / DATAFRAMES pour ce fichier ---
    diarization_list = []
    merged_list = []
    merged_list_g = []
    df_segments = pd.DataFrame(columns=["numero_segment", "start", "end", "text", "audio_file"])

    # --- TRANSCRIPTION ---
    audio = whisper.load_audio(audio_path)

    # Transcrire avec segmentation
    result = whisper_model.transcribe(audio_path,language="fr",condition_on_previous_text=False,no_speech_threshold=0.6,logprob_threshold=-1.0,compression_ratio_threshold=2.4)

    # Créer une liste de dict pour chaque segment
    segments_data = [
        {
            "start": seg["start"],
            "stop": seg["end"],
            "text": seg["text"].strip(),
            "audio_file": os.path.basename(audio_path)
        }
    for seg in result["segments"]
    ]

    # Transformer en DataFrame
    df_segments = pd.DataFrame(segments_data)

    # --- DIARIZATION ---
    data, sample_rate = sf.read(audio_path)
    waveform = torch.tensor(data.T, dtype=torch.float32)  # (channels, time)
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)  # mono → (1, time)

    audio_input = {"waveform": waveform, "sample_rate": sample_rate}
    output = pipeline(audio_input)

    for turn, speaker in output.speaker_diarization:
        diarization_list.append({
            "speaker": str(speaker),
            "start": turn.start,
            "end": turn.end
        })

    # Fusionner les segments consécutifs du même locuteur
    for seg_item in diarization_list:
        if not merged_list:
            merged_list.append(seg_item)
        else:
            last = merged_list[-1]
            if seg_item["speaker"] == last["speaker"]:
                last["end"] = seg_item["end"]
            else:
                merged_list.append(seg_item)
    
    ### MAIN SPEAKER
    # Détecter tous les locuteurs
    speakers = sorted({s["speaker"] for s in merged_list})

    # Ajouter colonnes pour chaque speaker
    for spk in speakers:
        df_segments[spk] = 0.0

    # Ajouter colonne pour le locuteur principal
    df_segments["main_speaker"] = None

    # Calcul du temps de parole et du locuteur principal
    for idx, row in df_segments.iterrows():
        seg_start = row["start"]
        seg_end = row["stop"]

    # dictionnaire temporaire pour stocker les durées
        speaker_times = {spk: 0.0 for spk in speakers}

        for s in merged_list:
            # intersection entre le segment et le sous-segment du speaker
            inter_start = max(seg_start, s["start"])
            inter_end = min(seg_end, s["end"])
            duration = max(0.0, inter_end - inter_start)
            if duration > 0:
                speaker_times[s["speaker"]] += duration

    # remplir les colonnes du DataFrame
        for spk, t in speaker_times.items():
            df_segments.at[idx, spk] = t

    # déterminer le locuteur principal
        main_spk = max(speaker_times, key=speaker_times.get)
        df_segments.at[idx, "main_speaker"] = main_spk


    # --- GENRE LOCUTEUR ---
    segmentation = seg_model(audio_path)

    for gender, start, stop in segmentation:
    
        if not merged_list_g:
            merged_list_g.append({"gender": gender, "start": start, "stop": stop})
    
        else:
            last = merged_list_g[-1]
        
            if gender == last["gender"]:
                last["stop"] = stop
        
            else:
                merged_list_g.append({"gender": gender, "start": start, "stop": stop})

    # Détecter tous les locuteurs
    gender = sorted({s["gender"] for s in merged_list_g})

    # Ajouter colonnes pour chaque speaker
    for g in gender:
        df_segments[g] = 0.0

    # Ajouter colonne pour le locuteur principal
    df_segments["main_g"] = None

    # Calcul du temps de parole et du locuteur principal
    for idx, row in df_segments.iterrows():
        seg_start = row["start"]
        seg_end = row["stop"]

        # dictionnaire temporaire pour stocker les durées
        gender_times = {g: 0.0 for g in gender}

        for s in merged_list_g:
            # intersection entre le segment et le sous-segment du speaker
            inter_start = max(seg_start, s["start"])
            inter_end = min(seg_end, s["stop"])
            duration = max(0.0, inter_end - inter_start)
            if duration > 0:
                gender_times[s["gender"]] += duration

        # remplir les colonnes du DataFrame
        for g, t in gender_times.items():
            df_segments.at[idx, g] = t

        # déterminer le locuteur principal
        main_g = max(gender_times, key=gender_times.get)
        df_segments.at[idx, "main_g"] = main_g

    # --- METTRE LE DF AU PROPRE ---
    # Fusionner les lignes d'un même locuteur
    df_segments['bloc'] = (df_segments['main_speaker'] != df_segments['main_speaker'].shift()).cumsum()

    df_segments = df_segments.groupby('bloc').agg(
        start=('start', 'first'),
        stop=('stop', 'last'),
        text=('text', ' '.join),
        main_speaker=('main_speaker', 'first'),
        main_g=('main_g', 'first'),
        audio_file=('audio_file', 'first')

    ).reset_index(drop=True)

    # --- ENREGISTRER LE RESULTAT EN CSV POUR CE FICHIER ---
    csv_name = os.path.splitext(os.path.basename(audio_path))[0] + "_transcription.csv"
    df_segments.to_csv("data/"+csv_name, index=False, encoding="utf-8-sig")
    df_segments.to_csv(f"s3://lab/PESSD/csv/{s3_folder}/{csv_name}", index=False, encoding="utf-8-sig")
    print(f"CSV créé pour {audio_path} : {csv_name}")

/opt/python/lib/python3.13/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/opt/python/lib/python3.13/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


In [9]:
pd.read_csv("data/PAF_wav_transcription.csv").to_csv("s3://lab/PESSD/csv/patin_art/PAF_wav_transcription.csv", index=False, encoding="utf-8-sig")